# Jamboree Admission — Exploratory Data Analysis

Explore the admissions data and identify preprocessing and modeling implications. The target is the continuous **Chance of Admit** value. The serial number is an identifier, not a predictive feature.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
DATA_PATH = Path("Jamboree_Admission.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd() / "Jamboree_Admission.csv"
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
print(f"Loaded {DATA_PATH.resolve()}")
df.head()

## Structure and data quality

Check the table dimensions, normalized column labels, inferred types, missingness, duplicate records, and basic statistics.

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().rename("missing").to_frame())
print("Duplicate rows:", int(df.duplicated().sum()))
display(df.describe().T)

## Target distribution

Inspect the range and shape of the admission-chance outcome before selecting a regression metric.

In [ ]:
TARGET = "Chance of Admit"
IDENTIFIER = "Serial No."
if TARGET not in df.columns:
    raise KeyError(f"Expected target column {TARGET!r}; found {df.columns.tolist()}")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[TARGET], kde=True, ax=axes[0], color="#4C78A8")
axes[0].set_title("Chance of Admit distribution")
sns.boxplot(x=df[TARGET], ax=axes[1], color="#72B7B2")
axes[1].set_title("Chance of Admit spread")
plt.tight_layout()
plt.show()
df[TARGET].describe()

## Feature distributions and relationships

Exclude the row identifier from feature analysis. These plots help spot scale differences, discrete variables, and nonlinear relationships.

In [ ]:
features = df.drop(columns=[TARGET, IDENTIFIER], errors="ignore")
fig, axes = plt.subplots(len(features.columns), 1, figsize=(10, 3 * len(features.columns)))
for column, ax in zip(features.columns, np.atleast_1d(axes)):
    sns.histplot(data=df, x=column, kde=(df[column].nunique() > 10), ax=ax, color="#59A14F")
    ax.set_title(f"Distribution: {column}")
plt.tight_layout()
plt.show()

plot_columns = list(features.columns)
ncols = 3
nrows = int(np.ceil(len(plot_columns) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
for column, ax in zip(plot_columns, np.ravel(axes)):
    sns.regplot(data=df, x=column, y=TARGET, ax=ax, scatter_kws={"alpha": 0.55, "s": 24}, line_kws={"color": "#E45756"})
    ax.set_title(f"{column} vs. {TARGET}")
for ax in np.ravel(axes)[len(plot_columns):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

## Correlations and modeling notes

Correlation is a useful first look at linear association, not proof of causation. The final cell produces data-dependent notes to guide the separate model-comparison script.

In [ ]:
analysis_df = df.drop(columns=[IDENTIFIER], errors="ignore")
corr = analysis_df.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True)
plt.title("Feature and target correlation matrix")
plt.tight_layout()
plt.show()

feature_target_corr = corr[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)
print("Absolute feature-target correlation ranking:")
display(feature_target_corr.to_frame("correlation"))
print(f"Rows: {len(df)}; numeric target: {pd.api.types.is_numeric_dtype(df[TARGET])}; missing cells: {int(df.isna().sum().sum())}")
print("Modeling notes:")
print("- Exclude Serial No. because it identifies row order rather than applicant merit.")
print("- Use a held-out test split and cross-validation on training data to compare regressors.")
print("- Fit any imputation/scaling inside a training-only pipeline to avoid leakage.")
print("- Compare MAE/RMSE alongside R-squared; do not interpret correlation as causation.")